# Predicting Student Test Scores 
## Score: 8.70787

In [1]:
import time
import hashlib
import numpy as np
import pandas as pd

import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')

test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')

bin_src_cols = [c for c in ['study_hours', 'sleep_hours', 'class_attendance'] if c in X.columns]
for c in bin_src_cols:
    _, bins = pd.qcut(X[c], q=20, duplicates='drop', retbins=True)
    bins[0] = -np.inf
    bins[-1] = np.inf
    bc = f'bin_{c}'
    X[bc] = pd.cut(X[c], bins=bins, include_lowest=True).cat.codes.astype('int16')
    X_test[bc] = pd.cut(X_test[c], bins=bins, include_lowest=True).cat.codes.astype('int16')


In [3]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'n_estimators': 8000,
    'num_leaves': 79,
    'max_depth': 10,
    'min_child_samples': 55,
    'reg_alpha': 10.0,
    'reg_lambda': 0.50,
    'min_split_gain': 1e-6,
    'subsample': 0.72,
    'subsample_freq': 3,
    'colsample_bytree': 0.65,
    'n_jobs': -1,
    'force_col_wise': True
}

seeds = [420, 666, 80085]
n_splits = 10

EARLY_STOP = 250
MAX_SECONDS = 3300

TE_SMOOTH = 25.0

te_cols = [c for c in ['course', 'exam_difficulty', 'study_method', 'sleep_quality', 'facility_rating', 'internet_access', 'gender'] if c in X.columns]
te_cols += [c for c in X.columns if str(c).startswith('bin_')]
te_cols = list(dict.fromkeys(te_cols))

te_pairs = []
for a, b in [('course', 'exam_difficulty'), ('study_method', 'exam_difficulty'), ('course', 'study_method')]:
    if a in X.columns and b in X.columns:
        te_pairs.append((a, b))

for c in te_cols:
    vc = pd.concat([X[c], X_test[c]]).value_counts(dropna=False)
    X[f'ce_{c}'] = X[c].map(vc).astype(float).fillna(0.0)
    X_test[f'ce_{c}'] = X_test[c].map(vc).astype(float).fillna(0.0)

t0 = time.time()

alt_params = {
    **base_params,
    'num_leaves': 47,
    'max_depth': -1,
    'min_child_samples': 90,
    'reg_alpha': 20.0,
    'reg_lambda': 1.50,
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.85
}


y_bins_cat = pd.qcut(pd.Series(y), q=20, duplicates='drop')
y_bins = y_bins_cat.cat.codes.to_numpy()
min_count = int(pd.Series(y_bins).value_counts().min())
if n_splits > min_count:
    n_splits = max(2, min_count)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def te_fit_1(X_ref, y_ref, col, smooth):
    y_s = pd.Series(y_ref, index=X_ref.index)
    g = y_s.groupby(X_ref[col], observed=False).agg(['mean', 'count'])
    prior = float(y_s.mean())
    enc = (g['mean'] * g['count'] + prior * smooth) / (g['count'] + smooth)
    return enc, prior

def te_fit_2(X_ref, y_ref, a, b, smooth):
    y_s = pd.Series(y_ref, index=X_ref.index)
    key = X_ref[a].astype(str) + '|' + X_ref[b].astype(str)
    g = y_s.groupby(key).agg(['mean', 'count'])
    prior = float(y_s.mean())
    enc = (g['mean'] * g['count'] + prior * smooth) / (g['count'] + smooth)
    return enc, prior

def te_apply_1(X_df, col, enc, prior):
    return X_df[col].map(enc).astype(float).fillna(prior)

def te_apply_2(X_df, a, b, enc, prior):
    key = X_df[a].astype(str) + '|' + X_df[b].astype(str)
    return key.map(enc).astype(float).fillna(prior)

def add_te(X_tr, X_va, X_te, y_tr):
    add_tr = {}
    add_va = {}
    add_te2 = {}

    for c in te_cols:
        enc, prior = te_fit_1(X_tr, y_tr, c, TE_SMOOTH)
        name = f'te_{c}'
        add_tr[name] = te_apply_1(X_tr, c, enc, prior)
        add_va[name] = te_apply_1(X_va, c, enc, prior)
        add_te2[name] = te_apply_1(X_te, c, enc, prior)

    for a, b in te_pairs:
        enc, prior = te_fit_2(X_tr, y_tr, a, b, TE_SMOOTH)
        name = f'te_{a}__{b}'
        add_tr[name] = te_apply_2(X_tr, a, b, enc, prior)
        add_va[name] = te_apply_2(X_va, a, b, enc, prior)
        add_te2[name] = te_apply_2(X_te, a, b, enc, prior)

    X_tr2 = pd.concat([X_tr, pd.DataFrame(add_tr, index=X_tr.index)], axis=1)
    X_va2 = pd.concat([X_va, pd.DataFrame(add_va, index=X_va.index)], axis=1)
    X_te2 = pd.concat([X_te, pd.DataFrame(add_te2, index=X_te.index)], axis=1)

    return X_tr2, X_va2, X_te2

def cv_run(params_list, seeds, label):
    sum_oof = np.zeros(len(X), dtype=float)
    cnt_oof = np.zeros(len(X), dtype=float)
    sum_test = np.zeros(len(X_test), dtype=float)
    seeds_done = 0

    for s_i, seed in enumerate(seeds, start=1):
        params_list2 = params_list if isinstance(params_list, (list, tuple)) else [params_list]

        oof = np.full(len(X), np.nan, dtype=float)
        test_pred_sum = np.zeros(len(X_test), dtype=float)
        rmse_scores = []
        folds_done = 0

        print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_bins), start=1):
            if (time.time() - t0) > MAX_SECONDS:
                break

            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)

            va_preds = []
            te_preds = []
            rmses = []

            for params in params_list2:
                p = {**params, 'random_state': seed}
                model = lgb.LGBMRegressor(**p)
                model.fit(
                    X_tr2,
                    y_tr,
                    eval_set=[(X_va2, y_va)],
                    callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(0)]
                )

                va_p = model.predict(X_va2)
                te_p = model.predict(X_test2)
                rmse_p = float(np.sqrt(mean_squared_error(y_va, va_p)))

                va_preds.append(va_p)
                te_preds.append(te_p)
                rmses.append(rmse_p)

            inv = 1.0 / (np.square(np.array(rmses, dtype=float)) + 1e-12)
            w = inv / inv.sum()

            va_pred = np.zeros(len(va_idx), dtype=float)
            te_pred = np.zeros(len(X_test), dtype=float)
            for wi, va_p, te_p in zip(w, va_preds, te_preds):
                va_pred += wi * va_p
                te_pred += wi * te_p

            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f} | w: {w.tolist()} | rmse: {rmses}')

            test_pred_sum += te_pred
            folds_done += 1

        if folds_done == 0:
            break

        test_pred = test_pred_sum / folds_done

        filled = ~np.isnan(oof)
        oof_filled = np.clip(oof[filled], 0, 100)
        sum_oof[filled] += oof_filled
        cnt_oof[filled] += 1.0

        sum_test += np.clip(test_pred, 0, 100)
        seeds_done += 1

        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], oof_filled)))
        print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        if (time.time() - t0) > MAX_SECONDS:
            break

    denom = np.maximum(cnt_oof, 1.0)
    all_oof = np.clip(sum_oof / denom, 0, 100)

    if seeds_done > 0:
        all_test = np.clip(sum_test / seeds_done, 0, 100)
    else:
        all_test = np.zeros(len(X_test), dtype=float)

    filled_all = cnt_oof > 0
    if filled_all.any():
        final_oof_rmse = float(np.sqrt(mean_squared_error(y[filled_all], all_oof[filled_all])))
    else:
        final_oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {final_oof_rmse:.5f}')

    return all_oof, all_test, final_oof_rmse


def cv_run_cb(params, seed, label, n_splits_cb=3):
    skf_cb = StratifiedKFold(n_splits=n_splits_cb, shuffle=True, random_state=seed)

    oof = np.full(len(X), np.nan, dtype=float)
    test_pred_sum = np.zeros(len(X_test), dtype=float)
    rmse_scores = []
    folds_done = 0

    cb_cat_cols = [c for c in te_cols if c in X.columns]

    print(f'{label} SEED {seed} (1/1)')

    for fold, (tr_idx, va_idx) in enumerate(skf_cb.split(X, y_bins), start=1):
        if (time.time() - t0) > MAX_SECONDS:
            break

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)

        cat_features = [int(X_tr2.columns.get_loc(c)) for c in cb_cat_cols if c in X_tr2.columns]

        tr_pool = Pool(X_tr2, y_tr, cat_features=cat_features)
        va_pool = Pool(X_va2, y_va, cat_features=cat_features)
        te_pool = Pool(X_test2, cat_features=cat_features)

        model = CatBoostRegressor(
            **params,
            random_seed=seed,
            allow_writing_files=False
        )

        print(f'  Fold {fold}/{n_splits_cb} start')

        model.fit(tr_pool, eval_set=va_pool, use_best_model=True, verbose=200)

        va_pred = model.predict(va_pool)
        te_pred = model.predict(te_pool)

        oof[va_idx] = va_pred
        test_pred_sum += te_pred
        folds_done += 1

        fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
        rmse_scores.append(fold_rmse)
        print(f'  Fold {fold}/{n_splits_cb} RMSE: {fold_rmse:.5f}')

    if folds_done == 0:
        all_test = np.full(len(X_test), np.nan, dtype=float)
    else:
        all_test = test_pred_sum / folds_done

    filled = ~np.isnan(oof)
    if filled.any():
        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], np.clip(oof[filled], 0, 100))))
    else:
        oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores) if rmse_scores else float("nan"):.5f} (+/- {np.std(rmse_scores) if rmse_scores else float("nan"):.5f})')

    return np.clip(oof, 0, 100), np.clip(all_test, 0, 100), oof_rmse


lgb_oof, lgb_test, _ = cv_run([base_params, alt_params], seeds, 'LGB2')

cb_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 2500,
    'learning_rate': 0.03,
    'depth': 8,
    'l2_leaf_reg': 6.0,
    'random_strength': 1.0,
    'bagging_temperature': 0.5,
    'subsample': 0.80,
    'rsm': 0.85,
    'min_data_in_leaf': 25,
    'od_type': 'Iter',
    'od_wait': 100,
    'thread_count': -1
}

cb_oof, cb_test, _ = cv_run_cb(cb_params, seed=2025, label='CAT', n_splits_cb=3)

cb_oof2 = np.array(cb_oof, dtype=float, copy=True)
cb_test2 = np.array(cb_test, dtype=float, copy=True)

m_oof = np.isnan(cb_oof2)
if m_oof.any():
    cb_oof2[m_oof] = lgb_oof[m_oof]

m_test = np.isnan(cb_test2)
if m_test.any():
    cb_test2[m_test] = lgb_test[m_test]

blend_X = np.vstack([lgb_oof, cb_oof2]).T
blend_lr = LinearRegression(positive=True)
blend_lr.fit(blend_X, y)

blend_oof = blend_lr.predict(blend_X)
blend_rmse = float(np.sqrt(mean_squared_error(y, blend_oof)))
print('BLEND coef', blend_lr.coef_.tolist(), 'intercept', float(blend_lr.intercept_), 'rmse', blend_rmse)

pred = blend_lr.predict(np.vstack([lgb_test, cb_test2]).T)
pred = np.clip(pred, 0, 100)

submission = pd.DataFrame({'id': test_ids, 'exam_score': pred})

out_path = 'submission.csv'
submission.to_csv(out_path, index=False)
with open(out_path, 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print(out_path)
print('md5', md5)
print('pred_mean', float(np.mean(pred)), 'pred_std', float(np.std(pred)), 'pred_min', float(np.min(pred)), 'pred_max', float(np.max(pred)))
print('pred_ge_99_5', int(np.sum(pred >= 99.5)), 'pred_ge_97', int(np.sum(pred >= 97.0)))


LGB2 SEED 420 (1/3)
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1492]	valid_0's rmse: 8.74633
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[2758]	valid_0's rmse: 8.73752
  Fold 1/10 RMSE: 8.73742 | w: [0.4994963158999551, 0.5005036841000449] | rmse: [8.74632954945635, 8.737523208580452]
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1432]	valid_0's rmse: 8.76328
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[2698]	valid_0's rmse: 8.76029
  Fold 2/10 RMSE: 8.75733 | w: [0.4998295213592498, 0.5001704786407503] | rmse: [8.763275269668963, 8.760287876355905]
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[2045]	valid_0's rmse: 8.7495
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[3310]	